# 03 - Modelagem Gold
Cria dimensão, fato, agregações de ativo/setor e ranking explicável.

In [0]:
%run ./00_setup

# 00 - Configuração
Cria objetos do Unity Catalog e caminhos do MVP. Ajuste os widgets antes da primeira execução.

Envie COTAHIST_A2025.TXT para: /Volumes/workspace/mvp_b3/landing/cotahist/
Envie setores_b3.csv para: /Volumes/workspace/mvp_b3/landing/referencia/


In [0]:
from pyspark.sql import functions as F, Window

q = spark.table(f"{catalog}.{schema}.silver_cotacoes")
# A referência contém o código-base atual do emissor (ex.: PETR) e também
# tickers históricos completos (ex.: ELET3). A correspondência exata tem
# precedência; o código-base é usado como fallback para os demais ativos.
e = spark.table(f"{catalog}.{schema}.silver_empresas")
e_exact = e.select(
    F.col("ticker").alias("ticker_ref_exato"),
    F.col("empresa").alias("empresa_exata"),
    F.col("setor").alias("setor_exato"),
    F.col("subsetor").alias("subsetor_exato"),
    F.col("segmento").alias("segmento_exato"),
)
e_base = e.select(
    F.col("ticker").alias("codigo_emissor"),
    F.col("empresa").alias("empresa_base"),
    F.col("setor").alias("setor_base"),
    F.col("subsetor").alias("subsetor_base"),
    F.col("segmento").alias("segmento_base"),
)
w = Window.partitionBy("ticker").orderBy("data_pregao")
w50 = w.rowsBetween(-49, 0)
w200 = w.rowsBetween(-199, 0)

fato = (q
    .withColumn("fechamento_anterior", F.lag("preco_fechamento").over(w))
    .withColumn("retorno_diario", F.col("preco_fechamento") / F.col("fechamento_anterior") - 1)
    .withColumn("mm_50", F.avg("preco_fechamento").over(w50))
    .withColumn("mm_200", F.avg("preco_fechamento").over(w200))
    .select("data_pregao", "ticker", "preco_abertura", "preco_maximo", "preco_minimo", "preco_fechamento",
            "numero_negocios", "quantidade_titulos", "volume_financeiro", "retorno_diario", "mm_50", "mm_200"))

(fato.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.fato_cotacao_diaria"))

last_w = Window.partitionBy("ticker").orderBy(F.col("data_pregao").desc())
latest = q.withColumn("rn", F.row_number().over(last_w)).filter("rn=1").drop("rn")
latest = latest.withColumn("codigo_emissor", F.substring("ticker", 1, 4))
dim = (latest
       .join(e_exact, latest.ticker == e_exact.ticker_ref_exato, "left")
       .join(e_base, "codigo_emissor", "left")
       .select(
           "ticker", "nome_resumido", "especificacao", "isin",
           F.coalesce("empresa_exata", "empresa_base").alias("empresa"),
           F.coalesce("setor_exato", "setor_base", F.lit("NAO_INFORMADO")).alias("setor"),
           F.coalesce("subsetor_exato", "subsetor_base", F.lit("NAO_INFORMADO")).alias("subsetor"),
           F.coalesce("segmento_exato", "segmento_base", F.lit("NAO_INFORMADO")).alias("segmento"),
       ))
(dim.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.dim_ativo"))

first_last = (fato.groupBy("ticker").agg(
    F.min_by("preco_fechamento", "data_pregao").cast("double").alias("primeiro_fechamento"),
    F.max_by("preco_fechamento", "data_pregao").cast("double").alias("ultimo_fechamento"),
    F.max_by("mm_50", "data_pregao").alias("ultimo_mm_50"),
    F.max_by("mm_200", "data_pregao").alias("ultimo_mm_200"),
    F.stddev_samp("retorno_diario").alias("std_retorno_diario"),
    F.avg("volume_financeiro").cast("double").alias("volume_medio_diario"),
    F.countDistinct("data_pregao").alias("numero_pregoes")))

indicadores = (first_last
    .withColumn("retorno_12m", F.col("ultimo_fechamento") / F.col("primeiro_fechamento") - 1)
    .withColumn("volatilidade_anual", F.col("std_retorno_diario") * F.sqrt(F.lit(252.0)))
    .withColumn("tendencia_50", (F.col("ultimo_fechamento") > F.col("ultimo_mm_50")).cast("int"))
    .withColumn("tendencia_200", (F.col("ultimo_fechamento") > F.col("ultimo_mm_200")).cast("int"))
    .drop("std_retorno_diario"))

(indicadores.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.agg_indicadores_ativos"))

# Elegibilidade: cobertura mínima e top 50% em liquidez, calculado dentro do próprio universo.
liq_cut = indicadores.approxQuantile("volume_medio_diario", [0.5], 0.01)[0]
elegiveis = (indicadores.filter((F.col("numero_pregoes") >= 180) & (F.col("volume_medio_diario") >= liq_cut))
             .join(dim.select("ticker", "setor"), "ticker"))

setores = elegiveis.groupBy("setor").agg(
    F.count("*").alias("quantidade_ativos"),
    F.sum("volume_medio_diario").alias("volume_medio_diario"),
    F.percentile_approx("retorno_12m", 0.5).alias("retorno_mediano_12m"),
    F.percentile_approx("volatilidade_anual", 0.5).alias("volatilidade_mediana"),
    F.avg("tendencia_200").alias("pct_acima_mm200"))
(setores.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.agg_desempenho_setor"))

global_w = Window.orderBy(F.lit(1))
ranked = (elegiveis
    .withColumn("score_retorno", F.percent_rank().over(Window.orderBy("retorno_12m")))
    .withColumn("score_tendencia", F.percent_rank().over(Window.orderBy((F.col("ultimo_fechamento") / F.col("ultimo_mm_200") - 1))))
    .withColumn("score_liquidez", F.percent_rank().over(Window.orderBy("volume_medio_diario")))
    .withColumn("score_risco", 1 - F.percent_rank().over(Window.orderBy("volatilidade_anual")))
    .withColumn("score_final", F.round(100 * (0.30*F.col("score_retorno") + 0.25*F.col("score_tendencia") + 0.25*F.col("score_liquidez") + 0.20*F.col("score_risco")), 2)))
ranked = ranked.withColumn("posicao", F.row_number().over(Window.orderBy(F.col("score_final").desc(), "ticker")))
(ranked.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.ranking_ativos_analise"))

# Comentários essenciais no catálogo.
comments = {
    "dim_ativo": "Dimensão de ativos negociados, enriquecida com classificação setorial.",
    "fato_cotacao_diaria": "Cotações diárias e indicadores de janela por ticker.",
    "agg_indicadores_ativos": "Indicadores de retorno, risco, liquidez e tendência por ativo.",
    "agg_desempenho_setor": "Consolidação robusta dos ativos líquidos por setor.",
    "ranking_ativos_analise": "Triagem quantitativa acadêmica; não é recomendação de investimento."
}
for table, comment in comments.items():
    spark.sql(f"COMMENT ON TABLE `{catalog}`.`{schema}`.`{table}` IS '{comment}'")

display(spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`"))

# Evidência de cobertura do enriquecimento setorial.
display(
    dim.groupBy("setor")
       .agg(F.count("*").alias("quantidade_ativos"))
       .orderBy(F.col("quantidade_ativos").desc())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


database,tableName,isTemporary
mvp_b3,agg_desempenho_setor,false
mvp_b3,agg_indicadores_ativos,false
mvp_b3,bronze_cotahist_raw,false
mvp_b3,bronze_setores_raw,false
mvp_b3,dim_ativo,false
mvp_b3,fato_cotacao_diaria,false
mvp_b3,ranking_ativos_analise,false
mvp_b3,silver_cotacoes,false
mvp_b3,silver_empresas,false


setor,quantidade_ativos
CONSUMO CÍCLICO,104
BENS INDUSTRIAIS,71
FINANCEIRO,70
UTILIDADE PÚBLICA,69
MATERIAIS BÁSICOS,48
CONSUMO NÃO CÍCLICO,29
SAÚDE,23
TECNOLOGIA DA INFORMAÇÃO,16
"PETRÓLEO, GÁS E BIOCOMBUSTÍVEIS",14
COMUNICAÇÕES,10


In [0]:
from pyspark.sql import functions as F

dim_check = spark.table(
    f"{catalog}.{schema}.dim_ativo"
)

display(
    dim_check.agg(
        F.count("*").alias("linhas_dimensao"),
        F.countDistinct("ticker").alias("tickers_distintos"),
        F.sum(
            F.when(
                F.col("setor") == "NAO_INFORMADO",
                1
            ).otherwise(0)
        ).alias("ativos_sem_setor")
    )
)

display(
    dim_check
    .filter(
        F.col("setor") == "NAO_INFORMADO"
    )
    .select(
        "ticker",
        "nome_resumido",
        "especificacao",
        "isin"
    )
    .orderBy("ticker")
)

linhas_dimensao,tickers_distintos,ativos_sem_setor
457,457,0


ticker,nome_resumido,especificacao,isin
